# 13 — Dash: Building Interactive Dashboards
**Goal:** Build a multi-page interactive dashboard with Dash — layout, callbacks,
filters, and live data — and run it from this notebook.

In [1]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import dash
print('dash', dash.__version__)
import pandas as pd
import plotly.express as px

dash 4.3.0


## 1. What Dash is

Dash is a Python framework for building **analytical web apps** without writing
JavaScript. It wraps:

- **Flask** for the HTTP layer
- **React** for the UI components (rendered in the browser)
- **Plotly** for the charts

You write Python; the user opens a URL and gets an interactive dashboard.

## 2. The mental model

A Dash app has three parts:

```
Layout      what is on the page (html.Div, dcc.Graph, dcc.Dropdown, ...)
Callbacks   how the page reacts to user input
Data        loaded once at startup (or refreshed on a schedule)
```

Each callback has the form:
```python
@app.callback(Output('id', 'property'), Input('id', 'property'))
def handler(value):
    return new_value
```

## 3. Your first Dash app — a single chart

We build the app in this cell. To run it, execute the cell and open the URL
Dash prints. The cell blocks because the server runs in the foreground — that
is expected. To run it in the background instead, see the variation below.

In [2]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

df = pd.read_csv('data/clean/unified_daily.csv')
num_cols = df.select_dtypes('number').columns.tolist()
x_col = 'date' if 'date' in df.columns else df.columns[0]
y_col = num_cols[0] if num_cols else None
cat_col = next((c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() < 10), None)
print('x =', x_col, '| y =', y_col, '| cat =', cat_col)

x = date | y = visits | cat = None


In [3]:
def build_minimal_app():
    app = dash.Dash(__name__)
    app.layout = html.Div([
        html.H1('Minimal Dash app'),
        dcc.Dropdown(
            id='channel',
            options=[{'label': c, 'value': c} for c in (df[cat_col].unique() if cat_col else [])],
            value=(df[cat_col].unique()[0] if cat_col else None),
            clearable=False,
        ),
        dcc.Graph(id='chart'),
    ])

    @app.callback(
        Output('chart', 'figure'),
        Input('channel', 'value'),
    )
    def update(channel):
        if not cat_col or not y_col:
            return px.line(title='No numeric column to plot')
        sub = df[df[cat_col] == channel]
        return px.line(sub, x=x_col, y=y_col, title=f'{y_col} — {channel}')

    return app

app = build_minimal_app()
print('app built. To run: app.run(debug=True, port=8050)')

app built. To run: app.run(debug=True, port=8050)


In [4]:
if False:    # flip to True to actually start the server (blocks)
    app.run(debug=True, port=8050, jupyter_mode='inline')

## 4. Building blocks — components

```
Layout
├── html.Div / html.H1 / html.P / html.Label
├── dcc.Graph            — Plotly chart
├── dcc.Dropdown         — single/multi select
├── dcc.RadioItems       — radio buttons
├── dcc.Slider / RangeSlider
├── dcc.DatePickerRange  — date filter
├── dcc.Tabs / dcc.Tab   — tabbed pages
├── dcc.Interval         — periodic timer
└── dcc.Store            — invisible state (for sharing data between callbacks)
```

In [5]:
print('Quick reference:')
for c in ['dcc.Graph', 'dcc.Dropdown', 'dcc.RadioItems', 'dcc.Slider',
          'dcc.RangeSlider', 'dcc.DatePickerRange', 'dcc.Tabs', 'dcc.Interval', 'dcc.Store']:
    print(' ', c)

Quick reference:
  dcc.Graph
  dcc.Dropdown
  dcc.RadioItems
  dcc.Slider
  dcc.RangeSlider
  dcc.DatePickerRange
  dcc.Tabs
  dcc.Interval
  dcc.Store


## 5. Layout — HTML structure + Bootstrap-style styling

Dash uses plain HTML for layout, not a grid system. We style with inline CSS
or a `assets/` folder that Dash serves automatically.

In [6]:
def shell(children, title='Dash demo'):
    return html.Div(
        style={'fontFamily': 'system-ui, sans-serif', 'margin': '20px'},
        children=[
            html.H1(title, style={'fontSize': 22, 'marginBottom': 4}),
            html.P('Sample dashboard built with Dash', style={'color': 'gray'}),
            html.Div(children),
        ],
    )

layout = shell([
    html.Div([
        dcc.Dropdown(id='x', options=[{'label': c, 'value': c} for c in num_cols],
                     value=num_cols[0] if num_cols else None),
        dcc.Graph(id='g'),
    ]),
])
print('layout constructed')

layout constructed


## 6. Multiple inputs and outputs in one callback

Callbacks can take multiple `Input`s and write to multiple `Output`s.

In [7]:
def multi_io_callback_demo():
    from dash.dependencies import Input, Output
    return (
        '@app.callback(\n'
        '    [Output("chart1", "figure"),\n'
        '     Output("chart2", "figure"),\n'
        '     Output("kpi", "children")]\n'
        '    [Input("channel-dropdown", "value"),\n'
        '     Input("date-range", "start_date"),\n'
        '     Input("date-range", "end_date")]\n'
        ')\n'
        'def update(c, start, end):\n'
        '    sub = df[(df["channel"] == c) &\n'
        '             (df["date"] >= start) &\n'
        '             (df["date"] <= end)]\n'
        '    fig1 = px.line(sub, x="date", y="spend")\n'
        '    fig2 = px.line(sub, x="date", y="conv")\n'
        '    kpi  = f"Conversions: {int(sub[\'conv\'].sum()):,}"\n'
        '    return fig1, fig2, kpi\n'
    )

print(multi_io_callback_demo())

@app.callback(
    [Output("chart1", "figure"),
     Output("chart2", "figure"),
     Output("kpi", "children")]
    [Input("channel-dropdown", "value"),
     Input("date-range", "start_date"),
     Input("date-range", "end_date")]
)
def update(c, start, end):
    sub = df[(df["channel"] == c) &
             (df["date"] >= start) &
             (df["date"] <= end)]
    fig1 = px.line(sub, x="date", y="spend")
    fig2 = px.line(sub, x="date", y="conv")
    kpi  = f"Conversions: {int(sub['conv'].sum()):,}"
    return fig1, fig2, kpi



## 7. Chained callbacks — dynamic options

When the options of one filter depend on another filter's value, chain
callbacks. The first writes to the second's `options`.

In [8]:
def chained_callback_demo():
    return (
        '@app.callback(\n'
        '    Output("channel-dropdown", "options"),\n'
        '    Input("region-dropdown", "value")\n'
        ')\n'
        'def set_channels(region):\n'
        '    channels = df[df["region"] == region]["channel"].unique()\n'
        '    return [{"label": c, "value": c} for c in channels]\n'
    )

print(chained_callback_demo())

@app.callback(
    Output("channel-dropdown", "options"),
    Input("region-dropdown", "value")
)
def set_channels(region):
    channels = df[df["region"] == region]["channel"].unique()
    return [{"label": c, "value": c} for c in channels]



## 8. Putting it together — a 4-tile growth dashboard

A small but realistic layout: KPI tiles on top, line + bar side by side,
breakdown table at the bottom.

In [9]:
def build_growth_app(data, kpi, grp, x_col):
    app = dash.Dash(__name__)
    channels = sorted(data[grp].dropna().unique()) if grp else []

    app.layout = html.Div(style={'fontFamily': 'system-ui', 'padding': '20px'}, children=[
        html.H2('Growth dashboard', style={'marginBottom': 0}),
        html.P('Source: unified marketing data', style={'color': 'gray'}),
        html.Div([
            dcc.Dropdown(id='ch', options=[{'label': c, 'value': c} for c in channels],
                         value=channels[0] if channels else None, clearable=False,
                         style={'width': '240px'}),
        ]),
        html.Div(id='kpis', style={'display': 'flex', 'gap': '12px', 'marginTop': '16px'}),
        html.Div([
            dcc.Graph(id='line', style={'flex': 2}),
            dcc.Graph(id='bar',  style={'flex': 1}),
        ], style={'display': 'flex', 'gap': '12px', 'marginTop': '16px'}),
        html.Div(id='breakdown', style={'marginTop': '16px'}),
    ])

    @app.callback(
        [Output('kpis', 'children'),
         Output('line', 'figure'),
         Output('bar',  'figure'),
         Output('breakdown', 'children')],
        Input('ch', 'value'),
    )
    def update(ch):
        if not grp or not ch:
            empty = px.line(title='No channel column')
            return [], empty, empty, ''
        sub = data[data[grp] == ch].copy()

        def kpi(label, value):
            return html.Div([
                html.Div(label, style={'color': 'gray', 'fontSize': 12}),
                html.Div(value, style={'fontSize': 22, 'fontWeight': 'bold'}),
            ], style={'padding': '12px 16px', 'border': '1px solid #eee',
                      'borderRadius': '6px', 'minWidth': '160px'})

        tiles = [kpi('Rows', f'{len(sub):,}')]
        for col in num_cols[:3]:
            tiles.append(kpi(f'{col} (sum)', f'{sub[col].sum():,.0f}'))

        line_fig = px.line(sub, x=x_col, y=num_cols[0] if num_cols else None,
                           title=f'{num_cols[0]} over time' if num_cols else 'data')
        bar_fig  = px.bar(sub.head(20), x=x_col,
                          y=(num_cols[1] if len(num_cols) > 1 else num_cols[0]),
                          title='Recent values')

        breakdown = html.Details([
            html.Summary('Show raw data'),
            dash.dash_table.DataTable(
                data=sub.head(50).to_dict('records'),
                columns=[{'name': c, 'id': c} for c in sub.columns],
                page_size=10, style_table={'overflowX': 'auto'},
            )
        ])
        return tiles, line_fig, bar_fig, breakdown
    return app

growth_app = build_growth_app(df, y_col, cat_col, x_col)
print('growth_app ready — run: growth_app.run(debug=True, port=8051)')

growth_app ready — run: growth_app.run(debug=True, port=8051)


In [10]:
if False:    # flip to True to actually start the server (blocks)
    growth_app.run(debug=True, port=8051, jupyter_mode='inline')

## 9. Running Dash as a script

In a real project, save the app in `src/app.py` and run:

```bash
uv run python src/app.py
```

```python
if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=8050)
```

Then visit `http://localhost:8050`.

In [11]:
import os
os.makedirs('src', exist_ok=True)
template = '''\
import os, pandas as pd, dash
from dash import dcc, html
from dash.dependencies import Input, Output

os.chdir(os.path.dirname(os.path.abspath(__file__)) + "/..")
df = pd.read_csv("data/clean/unified_daily.csv")

app = dash.Dash(__name__)
app.layout = html.Div([
    html.H1("Growth dashboard"),
    dcc.Dropdown(id="ch",
                 options=[{"label": c, "value": c} for c in df["channel"].unique()],
                 value=df["channel"].unique()[0]),
    dcc.Graph(id="g"),
])

@app.callback(Output("g", "figure"), Input("ch", "value"))
def update(ch):
    import plotly.express as px
    return px.line(df[df["channel"] == ch], x="date", y="spend")

if __name__ == "__main__":
    app.run(debug=True, host="0.0.0.0", port=8050)
'''
out = 'src/dash_growth_app.py'
if not os.path.exists(out):
    with open(out, 'w') as f:
        f.write(template)
print('wrote', out)

wrote src/dash_growth_app.py


## 10. Deploying

Three practical options:

1. **Local script** — `uv run python src/app.py` for personal use.
2. **Docker** — pin Python version, copy code, `CMD ["python", "app.py"]`.
3. **Hosted** — Render, Railway, Hugging Face Spaces, AWS App Runner.

Dash is just a Flask app, so anything that runs Flask can host Dash.

## Summary

| Concept | Use |
|---|---|
| `dash.Dash` | App factory |
| `html.*` | Layout (Div, H1, P) |
| `dcc.*` | Interactive (Graph, Dropdown, Slider) |
| `@app.callback` | Reactivity — Output(s) <- Input(s) |
| Chained callbacks | Dynamic options / dependent filters |
| `assets/` folder | CSS / JS / images for styling |
| `app.run(...)` | Local server (debug=True for dev) |

**Next:** `14_capstone_growth_dashboard.ipynb` — the end-to-end project.